# Uniform and log-normal distributions
The continuous uniform distribution, then the log-normal and how to check for it with a Q-Q plot.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

# fixed seed: the same random data on every run
rng = np.random.default_rng(42)

## 1. The continuous uniform distribution U(5, 6)
Production time of one product, equally likely anywhere from 5 to 6 hours.

In [ ]:
# scipy: loc = a, scale = b - a
u = stats.uniform(loc=5, scale=1)
print("density inside the range:", u.pdf(5.3))
print("P(5.2 <= X <= 5.5):", round(u.cdf(5.5) - u.cdf(5.2), 4))
print("mean", u.mean(), "variance", round(u.var(), 4), "std", round(u.std(), 4))
print("F(5.75):", u.cdf(5.75))

In [ ]:
# 1000 simulated production times: the histogram is roughly flat
times = rng.uniform(5, 6, size=1000)
x = np.linspace(4.5, 6.5, 500)
fig = go.Figure()
fig.add_histogram(x=times, nbinsx=20, histnorm="probability density", name="1000 times")
fig.add_scatter(x=x, y=u.pdf(x), mode="lines", name="PDF")
fig.update_layout(template="simple_white", xaxis_title="hours")
fig.show()
# skewness close to 0, excess kurtosis close to -1.2
print(round(stats.skew(times), 3), round(stats.kurtosis(times), 3))

## 2. The log-normal distribution
PDFs with mu = 0 and three values of sigma. In scipy, `s` is sigma and `scale` is e to the mu.

In [ ]:
x = np.linspace(0.001, 4, 400)
fig = go.Figure()
for s in [0.25, 0.5, 1.0]:
    fig.add_scatter(x=x, y=stats.lognorm(s=s, scale=1).pdf(x), name=f"sigma = {s}")
fig.update_layout(template="simple_white", xaxis_title="x", yaxis_title="density")
fig.show()

In [ ]:
# comment lengths: ln(words) ~ N(3, 1)
ln = stats.lognorm(s=1, scale=np.exp(3))
print("f(20):", round(ln.pdf(20), 4))
print("P(X > 100):", round(ln.sf(100), 4))
print("median", round(ln.median(), 1), "mean", round(ln.mean(), 1))

## 3. Is it log-normal? Take the log, then a Q-Q plot

In [ ]:
# 1000 simulated comment lengths, rounded to whole words (at least 1)
# a fresh generator with seed 42, as in the Note figure
words = np.round(np.random.default_rng(42).lognormal(mean=3, sigma=1, size=1000)).clip(1)
logs = np.log(words)
print("skewness raw:", round(stats.skew(words), 2), " after log:", round(stats.skew(logs), 2))
(osm, osr), (slope, icpt, r) = stats.probplot(logs, dist="norm")
fig = make_subplots(rows=1, cols=3, subplot_titles=["words", "ln(words)", "Q-Q of ln(words)"])
fig.add_histogram(x=words, nbinsx=50, row=1, col=1)
fig.add_histogram(x=logs, nbinsx=30, row=1, col=2)
fig.add_scatter(x=osm, y=osr, mode="markers", marker_size=3, row=1, col=3)
fig.add_scatter(x=osm, y=slope * osm + icpt, mode="lines", row=1, col=3)
fig.update_layout(template="simple_white", showlegend=False)
fig.show()